In [14]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import time

In [15]:
mnist = fetch_openml('mnist_784', version=1, cache=True, parser='auto') #fetching MNIST dataset

# Prepare features and labels
X = mnist.data.values.astype('float32') / 255.0
y = mnist.target.values.astype('int')

# Split 60k train / 10k test
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=60000, test_size=10000, random_state=42)
print(f"Data ready: {X_train.shape[0]} train samples, {X_test.shape[0]} test samples.\n")


Data ready: 60000 train samples, 10000 test samples.



In [16]:
T_checkpoints = [0.1, 1, 2, 3, 4, 10, 30]
methods_rows = [
    'Vote', 'Avg. (unnorm)', 'Avg. (norm)', 
    'Last (unnorm)', 'Last (norm)', 
    'Rand. (unnorm)', 'Rand. (norm)', 
    'SupVec', 'Mistake'
]

In [17]:
def train_predict_linear(X_tr, y_bin, X_test, epochs):
    """Runs Binary Linear Perceptron using dynamically resizing numpy arrays"""
    capacity = 100000
    M_x_arr = np.empty((capacity, X_tr.shape[1]), dtype=np.float32)
    M_y_arr = np.empty(capacity, dtype=np.float32)
    mistake_count = 0
    supvec_indices = set()
    C = [0]

    # Fast Training Phase
    v_curr = np.zeros(X_tr.shape[1], dtype=np.float32)
    for epoch in range(epochs):
        for i in range(len(X_tr)):
            x_i = X_tr[i]
            y_i = y_bin[i]

            y_hat = 1 if np.dot(v_curr, x_i) >= 0 else -1

            if y_hat == y_i:
                C[-1] += 1
            else:
                v_curr += y_i * x_i
                if mistake_count >= capacity:
                    capacity *= 2
                    new_M_x = np.empty((capacity, X_tr.shape[1]), dtype=np.float32)
                    new_M_x[:mistake_count] = M_x_arr
                    M_x_arr = new_M_x
                    
                    new_M_y = np.empty(capacity, dtype=np.float32)
                    new_M_y[:mistake_count] = M_y_arr
                    M_y_arr = new_M_y
                
                M_x_arr[mistake_count] = x_i
                M_y_arr[mistake_count] = y_i
                mistake_count += 1
                supvec_indices.add(i)
                C.append(1)

    # Reconstruction & Prediction Phase
    if mistake_count == 0:
        return (np.zeros(len(X_test)),)*6 + (0, set())

    updates = M_x_arr[:mistake_count] * M_y_arr[:mistake_count][:, np.newaxis]
    V_mat = np.vstack([np.zeros(X_tr.shape[1]), np.cumsum(updates, axis=0)])
    C_vec = np.array(C)

    norms = np.linalg.norm(V_mat, axis=1)
    norms[norms == 0] = 1.0 
    V_mat_norm = V_mat / norms[:, np.newaxis]

    dots_u = np.dot(X_test, V_mat.T)
    dots_n = np.dot(X_test, V_mat_norm.T)
    signs = np.where(dots_u >= 0, 1, -1)

    score_vote = np.dot(signs, C_vec)
    score_avg_u = np.dot(dots_u, C_vec)
    score_avg_n = np.dot(dots_n, C_vec)
    score_last_u = dots_u[:, -1]
    score_last_n = dots_n[:, -1]

    prob_weights = C_vec / np.sum(C_vec)
    score_rand = np.dot(signs, prob_weights)

    return (score_vote, score_avg_u, score_avg_n, score_last_u, 
            score_last_n, score_rand, mistake_count, supvec_indices)

In [18]:
def train_predict_kernel(d, X_tr, y_bin, X_test, epochs):
    """Runs Binary Kernel Perceptron using dynamically resizing numpy arrays"""
    capacity = 100000
    M_x_arr = np.empty((capacity, X_tr.shape[1]), dtype=np.float32)
    M_y_arr = np.empty(capacity, dtype=np.float32)
    mistake_count = 0
    supvec_indices = set()
    C = [0]

    # Fast Training Phase
    for epoch in range(epochs):
        for i in range(len(X_tr)):
            x_i = X_tr[i]
            y_i = y_bin[i]

            if mistake_count == 0:
                v_dot_x = 0.0
            else:
                v_dot_x = np.dot(M_y_arr[:mistake_count], (np.dot(M_x_arr[:mistake_count], x_i) + 1.0) ** d)

            y_hat = 1 if v_dot_x >= 0 else -1

            if y_hat == y_i:
                C[-1] += 1
            else:
                if mistake_count >= capacity:
                    capacity *= 2
                    new_M_x = np.empty((capacity, X_tr.shape[1]), dtype=np.float32)
                    new_M_x[:mistake_count] = M_x_arr
                    M_x_arr = new_M_x
                    
                    new_M_y = np.empty(capacity, dtype=np.float32)
                    new_M_y[:mistake_count] = M_y_arr
                    M_y_arr = new_M_y

                M_x_arr[mistake_count] = x_i
                M_y_arr[mistake_count] = y_i
                mistake_count += 1
                supvec_indices.add(i)
                C.append(1)

    # Prediction Phase
    M_x = M_x_arr[:mistake_count]
    M_y = M_y_arr[:mistake_count]
    C_vec = np.array(C)

    if mistake_count == 0:
         return (np.zeros(len(X_test)),)*6 + (0, set())

    batch_size = 1000
    score_vote, score_avg_u, score_avg_n = [], [], []
    score_last_u, score_last_n, score_rand = [], [], []
    prob_weights = C_vec / np.sum(C_vec)

    for b in range(0, len(X_test), batch_size):
        X_batch = X_test[b:b+batch_size]

        K_matrix = (np.dot(X_batch, M_x.T) + 1.0) ** d
        K_weighted = K_matrix * M_y

        H_vals = np.hstack([np.zeros((len(X_batch), 1)), np.cumsum(K_weighted, axis=1)])
        signs = np.where(H_vals >= 0, 1, -1)

        score_vote.extend(np.dot(signs, C_vec))
        score_last_u.extend(H_vals[:, -1])
        score_last_n.extend(H_vals[:, -1])
        score_avg_u.extend(np.dot(H_vals, C_vec)) 
        score_avg_n.extend(np.dot(H_vals, C_vec))
        score_rand.extend(np.dot(signs, prob_weights))

    return (np.array(score_vote), np.array(score_avg_u), np.array(score_avg_n),
            np.array(score_last_u), np.array(score_last_n), np.array(score_rand),
            mistake_count, supvec_indices)

In [19]:
def run_experiment_for_digit(target_digit, X_train, y_train, X_test, y_test, T_checkpoints, methods_rows):
    """Evaluates the perceptron models for a specific digit vs. all others."""
    
    y_train_bin = np.where(y_train == target_digit, 1, -1)
    y_test_bin = np.where(y_test == target_digit, 1, -1)
    
    table1 = pd.DataFrame(index=pd.MultiIndex.from_product([[1, 2, 3], methods_rows], names=['d', 'Method']), columns=T_checkpoints)
    table2 = pd.DataFrame(index=pd.MultiIndex.from_product([[4, 5, 6], methods_rows], names=['d', 'Method']), columns=T_checkpoints)

    for d in [1, 2, 3, 4, 5, 6]:
        print(f"\n[{target_digit} vs Rest] ======= Polynomial Degree d = {d} =======")
        
        for T in T_checkpoints:
            start_time = time.time()
            
            epochs = 1 if T == 0.1 else int(T)
            limit = 6000 if T == 0.1 else 60000
            
            X_tr = X_train[:limit]
            y_tr = y_train_bin[:limit]
            
            if d == 1:
                res = train_predict_linear(X_tr, y_tr, X_test, epochs)
            else:
                res = train_predict_kernel(d, X_tr, y_tr, X_test, epochs)
                
            scores_vote, scores_avg_u, scores_avg_n, scores_last_u, scores_last_n, scores_rand, mistakes_count, supvecs = res
            
            err_vote = (1 - accuracy_score(y_test_bin, np.where(scores_vote >= 0, 1, -1))) * 100
            err_avg_u = (1 - accuracy_score(y_test_bin, np.where(scores_avg_u >= 0, 1, -1))) * 100
            err_avg_n = (1 - accuracy_score(y_test_bin, np.where(scores_avg_n >= 0, 1, -1))) * 100
            err_last_u = (1 - accuracy_score(y_test_bin, np.where(scores_last_u >= 0, 1, -1))) * 100
            err_last_n = (1 - accuracy_score(y_test_bin, np.where(scores_last_n >= 0, 1, -1))) * 100
            err_rand = (1 - accuracy_score(y_test_bin, np.where(scores_rand >= 0, 1, -1))) * 100
            
            target_table = table1 if d <= 3 else table2
            target_table.loc[(d, 'Vote'), T] = round(err_vote, 1)
            target_table.loc[(d, 'Avg. (unnorm)'), T] = round(err_avg_u, 1)
            target_table.loc[(d, 'Avg. (norm)'), T] = round(err_avg_n, 1)
            target_table.loc[(d, 'Last (unnorm)'), T] = round(err_last_u, 1)
            target_table.loc[(d, 'Last (norm)'), T] = round(err_last_n, 1)
            target_table.loc[(d, 'Rand. (unnorm)'), T] = round(err_rand, 1)
            target_table.loc[(d, 'Rand. (norm)'), T] = round(err_rand, 1)
            target_table.loc[(d, 'SupVec'), T] = len(supvecs)
            target_table.loc[(d, 'Mistake'), T] = mistakes_count
            
            print(f"  T={T:<4} | Mistakes: {mistakes_count:<6} | Test Error: {err_vote:.2f}%  ({time.time() - start_time:.1f}s)")
            
    return table1, table2

In [20]:
all_results = {}
pd.set_option('future.no_silent_downcasting', True)

# Loop through all 10 digits
for digit in range(10):
    print(f"\n{'='*60}")
    print(f" STARTING EXPERIMENT FOR DIGIT: {digit}")
    print(f"{'='*60}")
    
    t1, t2 = run_experiment_for_digit(digit, X_train, y_train, X_test, y_test, T_checkpoints, methods_rows)
    all_results[digit] = {'table1': t1, 'table2': t2}
    
    print(f"\n=================== DIGIT {digit}: TABLE 1 (d=1, 2, 3) ===================")
    display(t1.fillna('-'))
    print(f"\n=================== DIGIT {digit}: TABLE 2 (d=4, 5, 6) ===================")
    display(t2.fillna('-'))


 STARTING EXPERIMENT FOR DIGIT: 0

[0 vs Rest] ======= Polynomial Degree d = 1 =======
  T=0.1  | Mistakes: 188    | Test Error: 1.24%  (0.1s)
  T=1    | Mistakes: 1119   | Test Error: 0.94%  (0.6s)
  T=2    | Mistakes: 2042   | Test Error: 0.92%  (1.2s)
  T=3    | Mistakes: 2916   | Test Error: 0.91%  (1.8s)
  T=4    | Mistakes: 3740   | Test Error: 0.89%  (2.3s)
  T=10   | Mistakes: 8393   | Test Error: 0.87%  (5.3s)
  T=30   | Mistakes: 22974  | Test Error: 0.85%  (14.8s)

[0 vs Rest] ======= Polynomial Degree d = 2 =======
  T=0.1  | Mistakes: 132    | Test Error: 0.92%  (0.8s)
  T=1    | Mistakes: 553    | Test Error: 0.32%  (11.4s)
  T=2    | Mistakes: 813    | Test Error: 0.24%  (24.7s)
  T=3    | Mistakes: 992    | Test Error: 0.19%  (35.7s)
  T=4    | Mistakes: 1160   | Test Error: 0.20%  (49.2s)
  T=10   | Mistakes: 1604   | Test Error: 0.19%  (155.3s)
  T=30   | Mistakes: 1843   | Test Error: 0.20%  (550.8s)

[0 vs Rest] ======= Polynomial Degree d = 3 =======
  T=0.1  | Mi

0.1   1.0   2.0   3.0   4.0   10.0   30.0
d Method                                                  
1 Vote            1.2   0.9   0.9   0.9   0.9   0.9    0.8
  Avg. (unnorm)   1.2   0.9   0.9   0.9   0.9   0.9    0.8
  Avg. (norm)     1.2   1.0   0.9   0.9   0.9   0.9    0.8
  Last (unnorm)   2.1   1.3   1.5   1.3   1.2   1.1    1.0
  Last (norm)     2.1   1.3   1.5   1.3   1.2   1.1    1.0
  Rand. (unnorm)  1.2   0.9   0.9   0.9   0.9   0.9    0.8
  Rand. (norm)    1.2   0.9   0.9   0.9   0.9   0.9    0.8
  SupVec          188  1119  1432  1625  1764  2225   2725
  Mistake         188  1119  2042  2916  3740  8393  22974
2 Vote            0.9   0.3   0.2   0.2   0.2   0.2    0.2
  Avg. (unnorm)   0.9   0.3   0.2   0.2   0.2   0.2    0.2
  Avg. (norm)     0.9   0.3   0.2   0.2   0.2   0.2    0.2
  Last (unnorm)   1.2   0.8   0.4   0.6   0.4   0.3    0.2
  Last (norm)     1.2   0.8   0.4   0.6   0.4   0.3    0.2
  Rand. (unnorm)  0.9   0.3   0.2   0.2   0.2   0.2    0.2
  Rand. (norm)    0.9   0.3   0.2   0.2   0.2   0.2    0.2
  SupVec          132   553   702   777   846  1000   1059
  Mistake         132   553   813   992  1160  1604   1843
3 Vote            0.8   0.3   0.2   0.1   0.1   0.2    0.2
  Avg. (unnorm)   0.8   0.3   0.2   0.2   0.1   0.2    0.1
  Avg. (norm)     0.8   0.3   0.2   0.2   0.1   0.2    0.1
  Last (unnorm)   1.2   0.4   0.3   0.2   0.3   0.2    0.2
  Last (norm)     1.2   0.4   0.3   0.2   0.3   0.2    0.2
  Rand. (unnorm)  0.8   0.3   0.2   0.1   0.1   0.2    0.2
  Rand. (norm)    0.8   0.3   0.2   0.1   0.1   0.2    0.2
  SupVec          119   477   594   644   683   775    798
  Mistake         119   477   659   739   803   983   1036


=================== DIGIT 0: TABLE 2 (d=4, 5, 6) ===================


0.1  1.0  2.0  3.0  4.0  10.0 30.0
d Method                                           
4 Vote            0.8  0.3  0.2  0.2  0.2  0.2  0.1
  Avg. (unnorm)   0.7  0.2  0.2  0.2  0.2  0.2  0.1
  Avg. (norm)     0.7  0.2  0.2  0.2  0.2  0.2  0.1
  Last (unnorm)   1.3  0.4  0.3  0.2  0.2  0.1  0.1
  Last (norm)     1.3  0.4  0.3  0.2  0.2  0.1  0.1
  Rand. (unnorm)  0.8  0.3  0.2  0.2  0.2  0.2  0.1
  Rand. (norm)    0.8  0.3  0.2  0.2  0.2  0.2  0.1
  SupVec          123  475  566  613  653  700  700
  Mistake         123  475  602  673  735  814  814
5 Vote            0.8  0.3  0.2  0.2  0.2  0.2  0.2
  Avg. (unnorm)   0.9  0.2  0.2  0.2  0.2  0.2  0.1
  Avg. (norm)     0.9  0.2  0.2  0.2  0.2  0.2  0.1
  Last (unnorm)   1.7  0.5  0.2  0.2  0.2  0.2  0.2
  Last (norm)     1.7  0.5  0.2  0.2  0.2  0.2  0.2
  Rand. (unnorm)  0.8  0.3  0.2  0.2  0.2  0.2  0.2
  Rand. (norm)    0.8  0.3  0.2  0.2  0.2  0.2  0.2
  SupVec          147  486  583  629  641  658  658
  Mistake         147  486  609  669  695  722  722
6 Vote            0.8  0.4  0.3  0.2  0.3  0.2  0.2
  Avg. (unnorm)   0.9  0.3  0.3  0.2  0.2  0.2  0.2
  Avg. (norm)     0.9  0.3  0.3  0.2  0.2  0.2  0.2
  Last (unnorm)   2.2  0.5  0.2  0.3  0.3  0.2  0.2
  Last (norm)     2.2  0.5  0.2  0.3  0.3  0.2  0.2
  Rand. (unnorm)  0.8  0.4  0.3  0.2  0.3  0.2  0.2
  Rand. (norm)    0.8  0.4  0.3  0.2  0.3  0.2  0.2
  SupVec          162  517  612  650  658  668  668
  Mistake         162  517  634  687  703  716  716


 STARTING EXPERIMENT FOR DIGIT: 1

[1 vs Rest] ======= Polynomial Degree d = 1 =======
  T=0.1  | Mistakes: 147    | Test Error: 1.08%  (0.1s)
  T=1    | Mistakes: 992    | Test Error: 0.85%  (0.6s)
  T=2    | Mistakes: 1773   | Test Error: 0.82%  (1.1s)
  T=3    | Mistakes: 2571   | Test Error: 0.81%  (1.5s)
  T=4    | Mistakes: 3341   | Test Error: 0.79%  (2.0s)
  T=10   | Mistakes: 7639   | Test Error: 0.80%  (4.8s)
  T=30   | Mistakes: 21015  | Test Error: 0.80%  (13.7s)

[1 vs Rest] ======= Polynomial Degree d = 2 =======
  T=0.1  | Mistakes: 78     | Test Error: 0.72%  (0.6s)
  T=1    | Mistakes: 435    | Test Error: 0.45%  (10.2s)
  T=2    | Mistakes: 684    | Test Error: 0.34%  (23.7s)
  T=3    | Mistakes: 868    | Test Error: 0.34%  (38.9s)
  T=4    | Mistakes: 997    | Test Error: 0.31%  (52.0s)
  T=10   | Mistakes: 1498   | Test Error: 0.28%  (169.0s)
  T=30   | Mistakes: 1937   | Test Error: 0.28%  (575.2s)

[1 vs Rest] ======= Polynomial Degree d = 3 =======
  T=0.1  | Mi

...

0.1   1.0   2.0   3.0   4.0   10.0  30.0
d Method                                                 
4 Vote            2.3   1.0   0.9   0.9   0.8   0.7   0.7
  Avg. (unnorm)   2.2   1.0   0.8   0.9   0.8   0.7   0.7
  Avg. (norm)     2.2   1.0   0.8   0.9   0.8   0.7   0.7
  Last (unnorm)   4.1   1.4   0.8   1.0   0.9   0.7   0.7
  Last (norm)     4.1   1.4   0.8   1.0   0.9   0.7   0.7
  Rand. (unnorm)  2.3   1.0   0.9   0.9   0.8   0.7   0.7
  Rand. (norm)    2.3   1.0   0.9   0.9   0.8   0.7   0.7
  SupVec          256  1167  1416  1538  1616  1799  1806
  Mistake         256  1167  1539  1744  1892  2215  2232
5 Vote            2.3   0.9   0.9   0.7   0.7   0.7   0.8
  Avg. (unnorm)   2.2   0.9   0.9   0.8   0.8   0.7   0.8
  Avg. (norm)     2.2   0.9   0.9   0.8   0.8   0.7   0.8
  Last (unnorm)   3.2   1.3   1.0   0.8   0.9   0.8   0.8
  Last (norm)     3.2   1.3   1.0   0.8   0.9   0.8   0.8
  Rand. (unnorm)  2.3   0.9   0.9   0.7   0.7   0.7   0.8
  Rand. (norm)    2.3   0.9   0.9   0.7   0.7   0.7   0.8
  SupVec          239  1117  1395  1520  1610  1675  1678
  Mistake         239  1117  1489  1666  1798  1930  1938
6 Vote            2.7   1.1   0.9   0.8   0.8   0.8   0.8
  Avg. (unnorm)   2.6   1.1   0.9   0.8   0.8   0.8   0.8
  Avg. (norm)     2.6   1.1   0.9   0.8   0.8   0.8   0.8
  Last (unnorm)   3.8   1.5   0.9   0.8   0.8   0.8   0.8
  Last (norm)     3.8   1.5   0.9   0.8   0.8   0.8   0.8
  Rand. (unnorm)  2.7   1.1   0.9   0.8   0.8   0.8   0.8
  Rand. (norm)    2.7   1.1   0.9   0.8   0.8   0.8   0.8
  SupVec          262  1245  1531  1656  1693  1748  1748
  Mistake         262  1245  1625  1828  1895  2007  2010


 STARTING EXPERIMENT FOR DIGIT: 9

[9 vs Rest] ======= Polynomial Degree d = 1 =======
  T=0.1  | Mistakes: 503    | Test Error: 4.62%  (0.2s)
  T=1    | Mistakes: 4065   | Test Error: 4.12%  (1.3s)
  T=2    | Mistakes: 7839   | Test Error: 4.00%  (2.6s)
  T=3    | Mistakes: 11576  | Test Error: 3.92%  (3.9s)
  T=4    | Mistakes: 15279  | Test Error: 3.93%  (5.2s)
  T=10   | Mistakes: 37245  | Test Error: 3.90%  (12.7s)
  T=30   | Mistakes: 109880 | Test Error: 3.90%  (59.0s)

[9 vs Rest] ======= Polynomial Degree d = 2 =======
  T=0.1  | Mistakes: 321    | Test Error: 2.55%  (2.1s)
  T=1    | Mistakes: 1622   | Test Error: 1.27%  (17.4s)
  T=2    | Mistakes: 2444   | Test Error: 1.03%  (41.8s)
  T=3    | Mistakes: 3080   | Test Error: 1.04%  (71.8s)
  T=4    | Mistakes: 3595   | Test Error: 0.96%  (81.9s)
  T=10   | Mistakes: 5600   | Test Error: 0.90%  (279.8s)
  T=30   | Mistakes: 7592   | Test Error: 0.90%  (974.5s)

[9 vs Rest] ======= Polynomial Degree d = 3 =======
  T=0.1  | M

0.1   1.0   2.0    3.0    4.0    10.0    30.0
d Method                                                       
1 Vote             4.6   4.1   4.0    3.9    3.9    3.9     3.9
  Avg. (unnorm)    4.7   4.1   4.0    3.9    3.9    3.9     3.9
  Avg. (norm)      4.7   4.1   4.0    3.9    3.9    3.9     3.9
  Last (unnorm)   12.3   7.3   6.0    4.7    5.3    5.3     6.4
  Last (norm)     12.3   7.3   6.0    4.7    5.3    5.3     6.4
  Rand. (unnorm)   4.6   4.1   4.0    3.9    3.9    3.9     3.9
  Rand. (norm)     4.6   4.1   4.0    3.9    3.9    3.9     3.9
  SupVec           503  4065  5201   5822   6288   7671    9107
  Mistake          503  4065  7839  11576  15279  37245  109880
2 Vote             2.5   1.3   1.0    1.0    1.0    0.9     0.9
  Avg. (unnorm)    2.5   1.2   1.0    1.0    1.0    0.9     0.9
  Avg. (norm)      2.5   1.2   1.0    1.0    1.0    0.9     0.9
  Last (unnorm)    3.3   1.5   1.4    1.6    1.2    0.8     1.0
  Last (norm)      3.3   1.5   1.4    1.6    1.2    0.8     1.0
  Rand. (unnorm)   2.5   1.3   1.0    1.0    1.0    0.9     0.9
  Rand. (norm)     2.5   1.3   1.0    1.0    1.0    0.9     0.9
  SupVec           321  1622  2032   2269   2423   2863    3182
  Mistake          321  1622  2444   3080   3595   5600    7592
3 Vote             2.0   1.1   0.8    0.8    0.7    0.7     0.7
  Avg. (unnorm)    2.0   1.0   0.9    0.8    0.8    0.7     0.7
  Avg. (norm)      2.0   1.0   0.9    0.8    0.8    0.7     0.7
  Last (unnorm)    3.6   1.5   1.0    1.0    0.8    0.8     0.7
  Last (norm)      3.6   1.5   1.0    1.0    0.8    0.8     0.7
  Rand. (unnorm)   2.0   1.1   0.8    0.8    0.7    0.7     0.7
  Rand. (norm)     2.0   1.1   0.8    0.8    0.7    0.7     0.7
  SupVec           286  1319  1614   1771   1900   2190    2223
  Mistake          286  1319  1834   2144   2407   3085    3190


=================== DIGIT 9: TABLE 2 (d=4, 5, 6) ===================


0.1   1.0   2.0   3.0   4.0   10.0  30.0
d Method                                                 
4 Vote            2.1   1.0   0.8   0.7   0.7   0.7   0.7
  Avg. (unnorm)   2.0   1.0   0.8   0.7   0.7   0.7   0.7
  Avg. (norm)     2.0   1.0   0.8   0.7   0.7   0.7   0.7
  Last (unnorm)   2.9   1.8   1.0   0.8   0.8   0.8   0.7
  Last (norm)     2.9   1.8   1.0   0.8   0.8   0.8   0.7
  Rand. (unnorm)  2.1   1.0   0.8   0.7   0.7   0.7   0.7
  Rand. (norm)    2.1   1.0   0.8   0.7   0.7   0.7   0.7
  SupVec          259  1169  1424  1560  1635  1863  1885
  Mistake         259  1169  1556  1769  1908  2311  2362
5 Vote            2.3   1.0   0.8   0.8   0.8   0.8   0.8
  Avg. (unnorm)   2.3   0.9   0.9   0.8   0.7   0.7   0.8
  Avg. (norm)     2.3   0.9   0.9   0.8   0.7   0.7   0.8
  Last (unnorm)   2.8   1.2   0.9   0.7   0.9   0.8   0.8
  Last (norm)     2.8   1.2   0.9   0.7   0.9   0.8   0.8
  Rand. (unnorm)  2.3   1.0   0.8   0.8   0.8   0.8   0.8
  Rand. (norm)    2.3   1.0   0.8   0.8   0.8   0.8   0.8
  SupVec          283  1173  1401  1503  1622  1734  1734
  Mistake         283  1173  1475  1662  1853  2060  2060
6 Vote            2.4   1.0   0.9   0.8   0.8   0.8   0.8
  Avg. (unnorm)   2.5   1.0   0.9   0.8   0.8   0.8   0.8
  Avg. (norm)     2.5   1.0   0.9   0.8   0.8   0.8   0.8
  Last (unnorm)   3.0   1.3   1.2   0.8   0.8   0.8   0.8
  Last (norm)     3.0   1.3   1.2   0.8   0.8   0.8   0.8
  Rand. (unnorm)  2.4   1.0   0.9   0.8   0.8   0.8   0.8
  Rand. (norm)    2.4   1.0   0.9   0.8   0.8   0.8   0.8
  SupVec          286  1172  1462  1642  1725  1761  1761
  Mistake         286  1172  1539  1804  1962  2055  2055